# Análise exploratória — multi-format-etl

EDA das quatro fontes de entrada (CSV, JSON, XML, logs) e das saídas
**curated**, **gold** e do relatório de **data quality**.

- **Offline**: tudo lido de `data/` (sem rede); se o arquivo real do Kaggle/BCB
  não existir (clone novo), o código cai na fixture versionada.
- Pré-requisito: `python scripts/run_load.py` já executado (gera a
  quality_report).


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from multi_format_etl.config import load_settings

settings = load_settings()
RAW, CURATED = settings.raw_dir, settings.curated_dir
GOLD, QUALITY = settings.gold_dir, settings.quality_dir

pd.set_option("display.max_columns", 50)
pd.options.display.float_format = "{:,.2f}".format
print(f"raiz do projeto: {settings.project_root}")
print(f"raw={RAW} | curated={CURATED}")

raiz do projeto: /home/julio-cesar/Documents/Projetos/data-portifolio-projects/multi-format-etl
raw=/home/julio-cesar/Documents/Projetos/data-portifolio-projects/multi-format-etl/data/raw | curated=/home/julio-cesar/Documents/Projetos/data-portifolio-projects/multi-format-etl/data/curated


## 1. CSV — transações bancárias (Kaggle)

Fonte real: dataset *USA Banking Transactions 2023–2024* (5.389 linhas,
20 colunas), baixado por `scripts/fetch_kaggle.py`.

In [2]:
csv_path = RAW / "csv" / "Banking_Transactions_USA_2023_2024.csv"
if not csv_path.exists():
    csv_path = RAW / "csv" / "transactions.csv"
    print("Arquivo real ausente (clone novo?) — usando a fixture versionada.")

df = pd.read_csv(csv_path)
print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

5,389 linhas x 20 colunas


,Transaction_ID,Account_Number,Transaction_Date,Transaction_Amount,Merchant_Name,Transaction_Type,Category,City,Country,Payment_Method,Customer_Age,Customer_Gender,Customer_Occupation,Customer_Income,Account_Balance,Transaction_Status,Fraud_Flag,Discount_Applied,Loyalty_Points_Earned,Transaction_Description
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,IUPM04409079772781,2023-11-05 15:54:38,"3,198.94",Houston Group,Debit,Transport,Phoenix,USA,Online Transfer,55,Others,Quality manager,"80,466.03",350.28,Failed,No,True,304,Recently company detail form range a.
1,23b8c1e9-3924-46de-beb1-3b9046685257,BLAT22216107051843,2024-04-21 22:21:55,129.93,Anderson-Phillips,Credit,Grocery,Philadelphia,USA,Debit Card,26,Others,Civil Service fast streamer,"145,574.25","9,797.81",Pending,Yes,False,383,Anything son baby power heart will not up.
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,UTXA55295806601382,2023-07-17 13:25:56,"1,378.77",Jensen Group,Credit,Shopping,New York,USA,Debit Card,29,Others,"Pilot, airline","33,447.18","12,399.85",Failed,Yes,False,497,Form world around green bar environment pattern.
3,972a8469-1641-4f82-8b9d-2434e465e150,XICF70493862044851,2023-06-27 16:09:52,"1,119.94","Nelson, Gomez and Rodriguez",Credit,Healthcare,Dallas,USA,Online Transfer,60,Male,"Radiographer, therapeutic","108,801.45","16,057.64",Failed,Yes,True,495,Order evening source these opportunity trade i...
4,17fc695a-07a0-4a6e-8822-e8f36c031199,KOSW19711121259020,2024-03-26 23:45:31,"3,683.67",Caldwell Group,Credit,Entertainment,San Jose,USA,E-Wallet,29,Others,Diplomatic Services operational officer,"100,985.12","14,940.54",Failed,Yes,True,292,Exactly politics door suggest.


In [3]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 5389 entries, 0 to 5388
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Transaction_ID           5389 non-null   str    
 1   Account_Number           5389 non-null   str    
 2   Transaction_Date         5389 non-null   str    
 3   Transaction_Amount       5389 non-null   float64
 4   Merchant_Name            5389 non-null   str    
 5   Transaction_Type         5389 non-null   str    
 6   Category                 5389 non-null   str    
 7   City                     5389 non-null   str    
 8   Country                  5389 non-null   str    
 9   Payment_Method           5389 non-null   str    
 10  Customer_Age             5389 non-null   int64  
 11  Customer_Gender          5389 non-null   str    
 12  Customer_Occupation      5389 non-null   str    
 13  Customer_Income          5389 non-null   float64
 14  Account_Balance          5389 non-n

In [4]:
print("nulos:", int(df.isna().sum().sum()), "| duplicados:", int(df.duplicated().sum()))
df.describe()

nulos: 0 | duplicados: 0


,Transaction_Amount,Customer_Age,Customer_Income,Account_Balance,Loyalty_Points_Earned
count,"5,389.00","5,389.00","5,389.00","5,389.00","5,389.00"
mean,"2,504.65",44.02,"85,802.75","10,096.51",249.90
std,"1,426.75",15.24,"37,343.60","5,732.85",145.38
min,5.46,18.00,"20,028.94",101.72,0.00
25%,"1,283.34",31.00,"53,809.80","5,159.45",122.00
50%,"2,521.67",44.00,"85,636.38","10,077.78",251.00
75%,"3,715.92",57.00,"118,092.64","15,076.01",377.00
max,"4,999.54",70.00,"149,970.50","19,993.04",500.00


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["Transaction_Status"].value_counts().plot(kind="bar", ax=axes[0], title="Transações por status")
df["Fraud_Flag"].value_counts().plot(kind="bar", ax=axes[1], title="Flag de fraude")
for ax in axes:
    ax.set_ylabel("transações")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

/tmp/ipykernel_166963/194730916.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])
mensal = (
    df.assign(month=df["Transaction_Date"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(transacoes=("Transaction_ID", "size"), volume=("Transaction_Amount", "sum"))
)
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(mensal.index, mensal["transacoes"], color="steelblue", label="transações")
ax1.set_ylabel("transações")
ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx()
ax2.plot(mensal.index, mensal["volume"], color="darkorange", marker="o", ms=3, label="volume")
ax2.set_ylabel("volume total")
ax1.set_title("Transações e volume por mês")
plt.tight_layout()
plt.show()
mensal.tail()

/tmp/ipykernel_166963/953792817.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,transacoes,volume
month,,
2024-09,235,"624,711.47"
2024-10,219,"562,963.04"
2024-11,231,"552,905.20"
2024-12,235,"577,163.47"
2025-01,135,"339,889.97"


In [7]:
fig, ax = plt.subplots(figsize=(8, 3.5))
df["Transaction_Amount"].plot(kind="hist", bins=40, ax=ax, color="steelblue",
                              title="Distribuição do valor da transação")
ax.set_xlabel("valor")
plt.tight_layout()
plt.show()
df["Transaction_Amount"].describe()

/tmp/ipykernel_166963/1209133438.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


count   5,389.00
mean    2,504.65
std     1,426.75
min         5.46
25%     1,283.34
50%     2,521.67
75%     3,715.92
max     4,999.54
Name: Transaction_Amount, dtype: float64

In [8]:
# outliers por método de Tukey (IQR)
q1, q3 = df["Transaction_Amount"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = df[(df["Transaction_Amount"] < lo) | (df["Transaction_Amount"] > hi)]
print(f"IQR={iqr:,.2f} | cercas: [{lo:,.2f}, {hi:,.2f}]")
print(f"outliers: {len(out):,} de {len(df):,} ({len(out) / len(df):.1%})")
fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot(df["Transaction_Amount"], vert=False)
ax.set_title("Boxplot — valor da transação (outliers = bigodes)")
ax.set_xlabel("valor")
plt.tight_layout()
plt.show()
out.nlargest(5, "Transaction_Amount")[
    ["Transaction_ID", "Transaction_Date", "Transaction_Amount", "Transaction_Type", "Fraud_Flag"]
]

IQR=2,432.58 | cercas: [-2,365.53, 7,364.79]
outliers: 0 de 5,389 (0.0%)


/tmp/ipykernel_166963/3161477793.py:9: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(df["Transaction_Amount"], vert=False)
/tmp/ipykernel_166963/3161477793.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Transaction_ID,Transaction_Date,Transaction_Amount,Transaction_Type,Fraud_Flag


In [9]:
# taxa de fraude por canal (Payment_Method)
df["fraud"] = (df["Fraud_Flag"] == "Yes").astype(int)
canal = df.groupby("Payment_Method")["fraud"].agg(["size", "mean"])
canal.columns = ["transacoes", "taxa_fraude"]
canal = canal.sort_values("taxa_fraude", ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.5))
canal["taxa_fraude"].plot(kind="bar", ax=ax, color="darkred",
                          title="Taxa de fraude por canal de pagamento")
ax.set_ylabel("taxa")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
canal

/tmp/ipykernel_166963/923121061.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,transacoes,taxa_fraude
Payment_Method,,
Credit Card,1094,0.50
Debit Card,1091,0.50
Cash,1019,0.50
E-Wallet,1100,0.48
Online Transfer,1085,0.48


In [10]:
# taxa de fraude por mês
fraude_mes = df.groupby(df["Transaction_Date"].dt.to_period("M").astype(str))["fraud"].mean()
fig, ax = plt.subplots(figsize=(12, 3.5))
fraude_mes.plot(ax=ax, color="crimson", marker="o", ms=3)
ax.set_title("Taxa de fraude por mês")
ax.set_ylabel("taxa")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()
print(f"média geral: {df['fraud'].mean():.1%} | mín {fraude_mes.min():.1%} | máx {fraude_mes.max():.1%}")

média geral: 49.3% | mín 42.7% | máx 56.6%


/tmp/ipykernel_166963/3701270665.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# fraude por tipo e por categoria
por_tipo = df.groupby("Transaction_Type")["fraud"].agg(["size", "mean"])
por_tipo.columns = ["transacoes", "taxa_fraude"]
por_categoria = df.groupby("Category")["fraud"].agg(["size", "mean"]).sort_values(
    "mean", ascending=False
)
por_categoria.columns = ["transacoes", "taxa_fraude"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
por_tipo["taxa_fraude"].plot(kind="bar", ax=axes[0], color="darkred",
                             title="Taxa de fraude por tipo")
por_categoria["taxa_fraude"].plot(kind="bar", ax=axes[1], color="indianred",
                                  title="Taxa de fraude por categoria (14)")
for ax in axes:
    ax.set_ylabel("taxa")
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylim(0, 0.65)
plt.tight_layout()
plt.show()
display(por_tipo)
por_categoria

/tmp/ipykernel_166963/2934725637.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,transacoes,taxa_fraude
Transaction_Type,,
Credit,2681,0.49
Debit,2708,0.49


,transacoes,taxa_fraude
Category,,
Utilities,417,0.53
Grocery,389,0.53
Fitness,374,0.53
Healthcare,370,0.51
Shopping,390,0.51
Transport,367,0.50
Entertainment,408,0.49
Housing,347,0.49
Clothing,396,0.48


In [12]:
# correlação de Pearson com a flag de fraude (colunas numéricas)
corr = df.corr(numeric_only=True)["fraud"].drop("fraud").sort_values()
fig, ax = plt.subplots(figsize=(8, 3.5))
corr.plot(kind="barh", ax=ax, color="steelblue",
          title="Correlação de Pearson com a flag de fraude")
ax.set_xlabel("r")
plt.tight_layout()
plt.show()
corr.round(3)

/tmp/ipykernel_166963/524150301.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Account_Balance         -0.04
Customer_Age            -0.02
Discount_Applied        -0.01
Transaction_Amount       0.00
Loyalty_Points_Earned    0.01
Customer_Income          0.01
Name: fraud, dtype: float64

## 2. JSON — câmbio USD/BRL (PTAX do Banco Central)

Série **PTAX** (`olinda.bcb.gov.br` — CotacaoDolarPeriodo) com janela
longa baixada por
`python scripts/fetch_json.py --start 01/01/2021 --end 03/10/2026`
(1.446 observações, ~5,8 anos — base para sazonalidade). O payload cru
vem em camelCase (`cotacaoCompra`/`cotacaoVenda`/`dataHoraCotacao`); o
`rename` do SPEC traduz para os nomes canônicos e o `datahora` (ISO, não
ambíguo) é parseado já na camada curated.

In [13]:
from multi_format_etl.load import load_json
from multi_format_etl.transform import standardize

ptax = standardize(
    load_json(RAW / "json" / "bcb_ptax_usd.json"), "json", "bcb_ptax_usd.json"
)
ptax["datahora"] = pd.to_datetime(ptax["datahora"])
ptax = ptax.sort_values("datahora").set_index("datahora")
print(f"{len(ptax)} observações de {ptax.index.min():%d/%m/%Y} a {ptax.index.max():%d/%m/%Y}")
ptax.tail()

1446 observações de 04/01/2021 a 02/10/2026


,cotacao_compra,cotacao_venda,_source_format,_source_file
datahora,,,,
2026-09-28 13:03:11.358850,5.21,5.21,json,bcb_ptax_usd.json
2026-09-29 13:06:06.446305,5.22,5.22,json,bcb_ptax_usd.json
2026-09-30 13:11:44.783262,5.18,5.18,json,bcb_ptax_usd.json
2026-10-01 13:10:35.446900,5.21,5.21,json,bcb_ptax_usd.json
2026-10-02 13:03:16.256632,5.22,5.22,json,bcb_ptax_usd.json


In [14]:
fig, ax = plt.subplots(figsize=(12, 4))
ptax["cotacao_venda"].plot(ax=ax, color="seagreen", lw=1)
ax.set_title("Câmbio USD/BRL — PTAX diária (venda)")
ax.set_ylabel("BRL por USD")
ax.set_xlabel("")
plt.tight_layout()
plt.show()
ptax["cotacao_venda"].describe()

/tmp/ipykernel_166963/3192690442.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


count   1,446.00
mean        5.29
std         0.30
min         4.62
25%         5.06
50%         5.24
75%         5.49
max         6.21
Name: cotacao_venda, dtype: float64

In [15]:
mensal_ptax = ptax["cotacao_venda"].resample("ME").agg(["size", "min", "max", "mean"])
mensal_ptax.columns = ["dias", "min", "max", "media"]
mensal_ptax

,dias,min,max,media
datahora,,,,
2021-01-31,20,5.16,5.51,5.36
2021-02-28,18,5.34,5.53,5.42
2021-03-31,23,5.50,5.84,5.65
2021-04-30,20,5.37,5.71,5.56
2021-05-31,21,5.22,5.45,5.29
...,...,...,...,...
2026-06-30,21,5.02,5.21,5.13
2026-07-31,23,5.06,5.20,5.11
2026-08-31,21,5.07,5.22,5.15


In [16]:
# variação e volatilidade no período
variacao = (ptax["cotacao_venda"].iloc[-1] / ptax["cotacao_venda"].iloc[0] - 1) * 100
print(f"variação no período: {variacao:+.2f}% "
      f"({ptax['cotacao_venda'].iloc[0]:.2f} -> {ptax['cotacao_venda'].iloc[-1]:.2f})")
print(f"díaria: min={ptax['cotacao_venda'].min():.2f} ({ptax['cotacao_venda'].idxmin():%d/%m/%Y}) | "
      f"max={ptax['cotacao_venda'].max():.2f} ({ptax['cotacao_venda'].idxmax():%d/%m/%Y})")
mensal_ptax["variacao_media_%"] = mensal_ptax["media"].pct_change() * 100
mensal_ptax.tail(3)

variação no período: +1.19% (5.16 -> 5.22)
díaria: min=4.62 (04/04/2022) | max=6.21 (02/01/2025)


,dias,min,max,media,variacao_media_%
datahora,,,,,
2026-08-31,21,5.07,5.22,5.15,0.77
2026-09-30,21,5.09,5.22,5.14,-0.16
2026-10-31,2,5.21,5.22,5.22,1.38


In [17]:
# sazonalidade — média do câmbio por mês do ano (± desvio)
sazonal = ptax["cotacao_venda"].groupby(ptax.index.month).agg(media="mean", desvio="std")
sazonal.index = ["jan", "fev", "mar", "abr", "mai", "jun",
                 "jul", "ago", "set", "out", "nov", "dez"]
fig, ax = plt.subplots(figsize=(9, 3.5))
sazonal["media"].plot(kind="bar", yerr=sazonal["desvio"].fillna(0), ax=ax,
                      color="slateblue", capsize=3,
                      title="Sazonalidade — média mensal do câmbio (± desvio)")
ax.set_ylabel("BRL por USD")
ax.set_xlabel("")
plt.tight_layout()
plt.show()
sazonal.round(2)

/tmp/ipykernel_166963/3978857697.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,media,desvio
jan,5.39,0.36
fev,5.29,0.26
mar,5.29,0.31
abr,5.22,0.36
mai,5.17,0.26
jun,5.16,0.25
jul,5.26,0.27
ago,5.24,0.22
set,5.25,0.20
out,5.37,0.21


## 3. XML e logs — fixtures legadas

XML (extrato de clientes) e logs (eventos da aplicação) não têm API
pública equivalente; seguem como fixtures com schemas congelados no
`docs/SPEC.md`.

In [18]:
from multi_format_etl.load import load_file

xml_df = load_file(RAW / "xml" / "customers.xml")[1]
logs_df = load_file(RAW / "logs" / "transactions.log")[1]
print(f"XML: {xml_df.shape[0]} clientes x {xml_df.shape[1]} colunas")
print(f"LOGS: {logs_df.shape[0]} eventos x {logs_df.shape[1]} colunas")
xml_df

XML: 3 clientes x 3 colunas
LOGS: 4 eventos x 5 colunas


,customer_id,full_name,branch_id
0,C0001,Ana Silva,B001
1,C0002,Bruno Souza,B002
2,C0003,Carla Oliveira,B001


In [19]:
display(logs_df)
display(logs_df["status"].value_counts().rename("eventos"))
display(logs_df["type"].value_counts().rename("eventos"))

,timestamp,account,type,amount,status
0,2024-06-01T10:00:00,A00001,deposit,1000.00,completed
1,2024-06-01T11:30:00,A00002,withdrawal,250.50,completed
2,2024-06-02T09:15:00,A00001,payment,89.90,failed
3,2024-06-03T08:00:00,A00003,fee,12.50,completed


status
completed    3
failed       1
Name: eventos, dtype: int64

type
deposit       1
withdrawal    1
payment       1
fee           1
Name: eventos, dtype: int64

## 4. Data quality e camada gold

Resumo do `quality_report.json` (checks executados, descartes por estágio)
e das tabelas de negócio em `data/gold/<formato>/`.

In [20]:
report_path = QUALITY / "quality_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("status:", report["status"], "| gerado em:", report["generated_at"])
    resumo = pd.DataFrame(
        {
            fmt: {
                "linhas_curated": v["rows_curated"],
                "checks": len(v["quality_checks"]),
                "descartados": v["clean"]["rows_in"] - v["clean"]["rows_out"],
                "tabelas_gold": len(v["gold_tables"]),
            }
            for fmt, v in report["formats"].items()
        }
    ).T
    display(resumo)
else:
    print("Relatório ausente — rode: python scripts/run_load.py")

status: passed | gerado em: 2026-10-03T17:00:28


,linhas_curated,checks,descartados,tabelas_gold
csv,5389,11,0,2
json,1446,10,0,1
xml,3,3,0,1
logs,4,12,0,2


In [21]:
tabelas = sorted(GOLD.glob("*/*.parquet"))
print(f"{len(tabelas)} tabelas gold")
for p in tabelas:
    g = pd.read_parquet(p)
    print(f"  {p.parent.name}/{p.stem}: {g.shape[0]} linhas x {g.shape[1]} colunas")

csv_status = pd.read_parquet(GOLD / "csv" / "by_status.parquet")
csv_status

6 tabelas gold
  csv/by_month: 25 linhas x 4 colunas
  csv/by_status: 3 linhas x 4 colunas
  json/by_month: 70 linhas x 5 colunas
  logs/by_status: 2 linhas x 3 colunas
  logs/by_type: 4 linhas x 3 colunas
  xml/by_branch: 2 linhas x 2 colunas


,transaction_status,transactions,total_amount,avg_amount
0,Failed,1789,"4,456,909.77","2,491.29"
1,Pending,1811,"4,530,707.98","2,501.77"
2,Success,1789,"4,509,936.79","2,520.93"
